#### Notebook 11 — transformer_ticket_classifier

##### Purpose

Build and evaluate a fine-tuned Transformer classifier for support-ticket categorization.

The notebook extends previous approaches:

``` text

TF-IDF
  ↓
Logistic Regression

Pretrained MiniLM
  ↓
Fixed 384-d embeddings
  ↓
Logistic Regression / Neural Network

Pretrained MiniLM Transformer
  ↓
Fine-tuning
  ↓
4-class Transformer classifier

```

The main learning objective is to demonstrate that during full fine-tuning, the pretrained Transformer parameters themselves are updated by the classification loss.

##### 1. Technologies

- Apache Spark / Delta
- PyTorch
- Hugging Face Transformers
- MiniLM
- scikit-learn
- pandas

##### 2. Architecture

``` text

Support ticket
      ↓
BertTokenizerFast
      ↓
input_ids + attention_mask
      ↓
Pretrained MiniLM Transformer
      ↓
last_hidden_state
[batch, sequence_length, 384]
      ↓
Masked Mean Pooling
      ↓
[batch, 384]
      ↓
Dropout
      ↓
Linear(384, 4)
      ↓
4 logits
      ↓
CrossEntropyLoss
      ↓
Backpropagation
      ↓
Transformer + classifier parameters updated

```

Classes:

- 0 → Billing
- 1 → Cancellation
- 2 → Login
- 3 → Technical

##### 3. Imports

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from src.project_config import (
    RANDOM_SEED,
    CLASS_NAMES,
    TEXT_COL,
    TARGET_COL,
    TRANSFORMER_MODEL_NAME,
    TRANSFORMER_BATCH_SIZE,
    TRANSFORMER_MAX_LENGTH,
    TRANSFORMER_LEARNING_RATE,
    TRANSFORMER_MAX_EPOCHS,
    TRANSFORMER_EARLY_STOPPING_PATIENCE,
)

from src.data_preparation import (
    load_modeling_dataset,
    split_modeling_dataset,
)

from src.transformer_classifier import (
    set_transformer_seed,
    create_transformer_model,
    create_transformer_dataloader,
    count_parameters,
    fine_tune_transformer,
    evaluate_transformer,
)

##### 4. Reproducibility and compute device

In [0]:
set_transformer_seed(
    RANDOM_SEED
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)

##### 5. Load persisted modeling data

In [0]:
modeling_df = load_modeling_dataset(spark)

train_df, validation_df, test_df = (
    split_modeling_dataset(
        modeling_df
    )
)

print(
    "Train rows:",
    train_df.count()
)

print(
    "Validation rows:",
    validation_df.count()
)

print(
    "Test rows:",
    test_df.count()
)

##### 6. Spark → pandas boundary

In [0]:
train_pd = train_df.toPandas()
validation_pd = validation_df.toPandas()
test_pd = test_df.toPandas()

In [0]:
print(
    train_pd.shape,
    validation_pd.shape,
    test_pd.shape,
)

##### 7. Define class mapping

In [0]:
label_to_id = {
    label: index
    for index, label in enumerate(CLASS_NAMES)
}

id_to_label = {
    index: label
    for label, index in label_to_id.items()
}

label_to_id

##### 8. Generate target arrays:

In [0]:
y_train = (
    train_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

y_validation = (
    validation_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

y_test = (
    test_pd[TARGET_COL]
    .map(label_to_id)
    .to_numpy()
)

In [0]:
assert not pd.isna(y_train).any()
assert not pd.isna(y_validation).any()
assert not pd.isna(y_test).any()

print(
    "Train labels:",
    y_train.shape
)

print(
    "Validation labels:",
    y_validation.shape
)

print(
    "Test labels:",
    y_test.shape
)

##### 9. Load tokenizer and Transformer

In [0]:
tokenizer, model = (
    create_transformer_model(
        num_classes=len(
            CLASS_NAMES
        )
    )
)

print(tokenizer)
print(model)

In [0]:
print(
    "Transformer hidden size:",
    model.transformer.config.hidden_size
)

print(
    "Number of classes:",
    len(CLASS_NAMES)
)

##### 10. Verify full fine-tuning

In [0]:
parameter_info = count_parameters(model)

parameter_info

In [0]:
print(
    "Total parameters:",
    f"{parameter_info['total_parameters']:,}"
)

print(
    "Trainable parameters:",
    f"{parameter_info['trainable_parameters']:,}"
)

print(
    "Frozen parameters:",
    f"{parameter_info['frozen_parameters']:,}"
)

In [0]:
for name, parameter in list(
    model.named_parameters()
)[:10]:

    print(
        name,
        parameter.requires_grad
    )

##### 11. Create DataLoaders

In [0]:
train_loader = (
    create_transformer_dataloader(
        texts=train_pd[TEXT_COL],
        labels=y_train,
        tokenizer=tokenizer,
        batch_size=TRANSFORMER_BATCH_SIZE,
        shuffle=True,
    )
)

validation_loader = (
    create_transformer_dataloader(
        texts=validation_pd[TEXT_COL],
        labels=y_validation,
        tokenizer=tokenizer,
        batch_size=TRANSFORMER_BATCH_SIZE,
        shuffle=False,
    )
)

test_loader = (
    create_transformer_dataloader(
        texts=test_pd[TEXT_COL],
        labels=y_test,
        tokenizer=tokenizer,
        batch_size=TRANSFORMER_BATCH_SIZE,
        shuffle=False,
    )
)

##### 12. Inspect one training batch

In [0]:
sample_batch = next(
    iter(train_loader)
)

print(
    "input_ids:",
    sample_batch[
        "input_ids"
    ].shape
)

print(
    "attention_mask:",
    sample_batch[
        "attention_mask"
    ].shape
)

print(
    "labels:",
    sample_batch[
        "labels"
    ].shape
)

##### 13. Inspect Transformer representations

In [0]:
model = model.to(device)

input_ids = (
    sample_batch["input_ids"]
    .to(device)
)

attention_mask = (
    sample_batch["attention_mask"]
    .to(device)
)

with torch.no_grad():

    transformer_outputs = model.transformer(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

print(
    "Last hidden state:",
    transformer_outputs.last_hidden_state.shape
)

##### 14. Verify pooling

In [0]:
with torch.no_grad():

    pooled = model.mean_pool(
        transformer_outputs
        .last_hidden_state,
        attention_mask,
    )

print(
    "Pooled:",
    pooled.shape
)

##### 15. Verify classification logits

In [0]:
with torch.no_grad():

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

print(
    "Logits:",
    logits.shape
)

##### 16. Capture pretrained Transformer parameter

In [0]:
transformer_parameter_before = (
    model
    .transformer
    .encoder
    .layer[0]
    .attention
    .self
    .query
    .weight
    .detach()
    .cpu()
    .clone()
)

print(
    transformer_parameter_before.shape
)

##### 17. Fine-tune

In [0]:
model, training_history = (
    fine_tune_transformer(
        model=model,
        train_loader=train_loader,
        validation_loader=(
            validation_loader
        ),
        device=device,
        max_epochs=(
            TRANSFORMER_MAX_EPOCHS
        ),
        patience=(
            TRANSFORMER_EARLY_STOPPING_PATIENCE
        ),
    )
)

##### 18. Training history

In [0]:
history_df = pd.DataFrame(
    training_history
)

display(
    history_df
)

##### 19. Prove Transformer parameters changed

In [0]:
transformer_parameter_after = (
    model
    .transformer
    .encoder
    .layer[0]
    .attention
    .self
    .query
    .weight
    .detach()
    .cpu()
    .clone()
)

In [0]:
print(
    transformer_parameter_after.shape
)

In [0]:
parameter_change = (
    transformer_parameter_after
    -
    transformer_parameter_before
)

print(
    "Maximum absolute change:",
    parameter_change
    .abs()
    .max()
    .item()
)

print(
    "Mean absolute change:",
    parameter_change
    .abs()
    .mean()
    .item()
)

##### 20. Evaluate validation set

In [0]:
criterion = nn.CrossEntropyLoss()

validation_result = (
    evaluate_transformer(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=device,
    )
)

print(
    "Validation Accuracy:",
    round(
        validation_result[
            "accuracy"
        ],
        4,
    )
)

##### 21. Final test evaluation

In [0]:
test_result = (
    evaluate_transformer(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
    )
)

In [0]:
test_labels = (
    test_result["labels"]
)

test_predictions = (
    test_result["predictions"]
)

test_probabilities = (
    test_result["probabilities"]
)

In [0]:
test_accuracy = accuracy_score(
    test_labels,
    test_predictions,
)

test_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
)

test_weighted_f1 = f1_score(
    test_labels,
    test_predictions,
    average="weighted",
)

print(
    "Test Accuracy:",
    round(
        test_accuracy,
        4,
    )
)

print(
    "Macro F1:",
    round(
        test_macro_f1,
        4,
    )
)

print(
    "Weighted F1:",
    round(
        test_weighted_f1,
        4,
    )
)

##### 22. Classification report

In [0]:
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

##### 23. Confusion matrix

In [0]:
confusion = confusion_matrix(
    test_labels,
    test_predictions,
)

confusion_df = pd.DataFrame(
    confusion,
    index=[
        f"Actual_{label}"
        for label in CLASS_NAMES
    ],
    columns=[
        f"Predicted_{label}"
        for label in CLASS_NAMES
    ],
)

display(
    confusion_df
)

##### 24. Ticket-level predictions

In [0]:
prediction_df = (
    test_pd[
        [
            "ticket_id",
            TEXT_COL,
            TARGET_COL,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

prediction_df[
    "predicted_category"
] = [
    id_to_label[
        prediction
    ]
    for prediction in test_predictions
]

prediction_df[
    "confidence"
] = (
    test_probabilities.max(
        axis=1
    )
)

prediction_df[
    "correct"
] = (
    prediction_df[
        TARGET_COL
    ]
    ==
    prediction_df[
        "predicted_category"
    ]
)

display(
    prediction_df
)

confidence = maximum softmax probability

##### 25. Inspect Login tickets

In [0]:
display(
    prediction_df[
        prediction_df[
            TARGET_COL
        ]
        == "Login"
    ]
)

##### 26. Final verification

In [0]:
assert (
    model.transformer
    .config.hidden_size
    == 384
)

assert (
    test_probabilities.shape[1]
    == len(CLASS_NAMES)
)

assert (
    len(test_predictions)
    == len(test_pd)
)

assert (
    parameter_change
    .abs()
    .max()
    .item()
    > 0
)

print(
    "Transformer fine-tuning "
    "verification passed."
)

##### Key Learnings


1. Pretraining and fine-tuning are different stages

    - Pretraining → model learns general language representations from large external datasets
    - Fine-tuning → pretrained parameters are adapted for our support-ticket classification task

2. Full fine-tuning means the Transformer itself is trainable

    Our model contained:

    - 22,714,756 total parameters
    - 22,714,756 trainable parameters
    - 0 frozen parameters

Therefore the pretrained Transformer and new classification head were eligible for gradient-based updates.

3. Fine-tuning changes parameter values, not architecture

- Query matrix before: [384,384]
- Query matrix after:  [384,384]

The shape remained unchanged, but the values changed.

4. Transformer training follows the same fundamental algorithm as neural-network training

    - Forward → Loss → Backpropagation → Optimizer update

The major difference is that the network being updated is much larger and starts with pretrained knowledge.

5. Pooling reduces the token dimension

``` text

[batch, sequence_length, 384]
              ↓
         mean pooling
              ↓
        [batch,384]

```

The 384 feature dimensions remain.

6. Training, validation, and test have distinct responsibilities

    - Train → updates parameters
    - Validation → monitors/selects model → no parameter updates
    - Test → final evaluation → no parameter updates

7. Accuracy and loss measure different things

- Accuracy only asks whether the winning class is correct.
- CrossEntropyLoss also responds to the probability assigned to the correct class. Therefore validation loss can continue decreasing even after validation accuracy reaches 1.0.

8. Perfect test accuracy must be interpreted cautiously

The test set contains only 13 examples and repeated/highly similar ticket text. The result demonstrates that the pipeline works, but does not establish real-world Transformer performance.

9. More complex does not automatically mean better

Our frozen MiniLM embedding approaches already performed perfectly on this small test split. Full Transformer fine-tuning adds compute and overfitting risk.

The purpose of Notebook 11 was therefore primarily to learn and demonstrate Transformer fine-tuning, rather than prove it is the preferred production solution.

##### Conclusion

A strong final conclusion would be:

This notebook extended support-ticket classification from frozen pretrained embeddings to full Transformer fine-tuning. A pretrained MiniLM/BERT-style encoder was combined with masked mean pooling and a four-class classification head. All 22.7 million model parameters were trainable, allowing classification loss to propagate through the classifier and pretrained Transformer layers.

Fine-tuning was directly verified by comparing an attention query weight matrix before and after training. The matrix retained its [384,384] architecture while its parameter values changed, demonstrating task-specific adaptation of the pretrained Transformer.

The model achieved perfect accuracy and F1 scores on the small 13-ticket test split, including the Login examples previously missed by TF-IDF. Because the dataset is extremely small and contains repeated or highly similar text, these metrics should be treated as a pipeline validation and learning result rather than evidence of production-level generalization.

The experiment demonstrates the progression from sparse lexical features, to frozen semantic embeddings, to neural classification, and finally to end-to-end Transformer fine-tuning.

##### Next — Notebook 12: mlflow_tracking


For Notebook 12, we should track not only accuracy/F1, but also the pretrained model name, max sequence length, learning rate, batch size, epochs, trainable parameter count, training history, label mapping, model artifacts, and enough preprocessing/tokenizer information to reproduce inference.